In [8]:
import os
import glob
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

pd.set_option('display.max_columns', None)

In [9]:
# 1. Load sales data
data_path = "data/cleaned_sales_data.csv"
if not os.path.exists(data_path) and os.path.exists("../data/cleaned_sales_data.csv"):
    data_path = "../data/cleaned_sales_data.csv"

df = pd.read_csv(data_path, low_memory=False)

# Remove outliers
df = df[(df["ClosePrice"] >= 100000) & (df["ClosePrice"] <= 5000000)].copy()
df["CloseDate"] = pd.to_datetime(df["CloseDate"])

df["bed_bath_ratio"] = df["BedroomsTotal"] / (df["BathroomsTotalInteger"].fillna(1) + 1)
df["property_age"] = (2026 - df["YearBuilt"]).apply(lambda x: x if 0 <= x <= 150 else np.nan)
df["property_age"] = df["property_age"].fillna(df["property_age"].median())

# 3. Spatial join with CA School Districts
shapefile_matches = glob.glob("data/school_districts/*.shp") + glob.glob("../data/school_districts/*.shp")
shapefile_path = shapefile_matches[0]
school_gdf = gpd.read_file(shapefile_path)

df_geo = df.dropna(subset=["Latitude", "Longitude"]).copy()
geometry = [Point(xy) for xy in zip(df_geo["Longitude"], df_geo["Latitude"])]
properties_gdf = gpd.GeoDataFrame(df_geo, geometry=geometry, crs="EPSG:4326")

school_gdf = school_gdf.to_crs(properties_gdf.crs)
district_col = [col for col in school_gdf.columns if "NAME" in col.upper() or "DISTRICT" in col.upper()][0]
joined_gdf = gpd.sjoin(properties_gdf, school_gdf[[district_col, "geometry"]], how="left", predicate="within")

joined_gdf = joined_gdf[~joined_gdf.index.duplicated(keep="first")]
df["school_district"] = "Unknown"
df.loc[joined_gdf.index, "school_district"] = joined_gdf[district_col].fillna("Unknown")

district_means = df.groupby("school_district")["ClosePrice"].transform("mean")
df["school_district_avg_price"] = district_means

In [10]:
feature_cols_raw = [
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger", 
    "LotSizeSquareFeet", "bed_bath_ratio", "property_age", "school_district_avg_price"
]

for col in feature_cols_raw:
    df[col] = df[col].fillna(df[col].median())

scaler = StandardScaler()
df[[f"{col}_scaled" for col in feature_cols_raw]] = scaler.fit_transform(df[feature_cols_raw])

scaled_feature_cols = [f"{col}_scaled" for col in feature_cols_raw]

test_mask = (df["CloseDate"].dt.year == 2026) & (df["CloseDate"].dt.month == 5)
test_df = df[test_mask].copy()

test_start = pd.Timestamp("2026-05-01")
train_start = test_start - pd.DateOffset(months=6)
train_df = df[(df["CloseDate"] >= train_start) & (df["CloseDate"] < test_start)].copy()

X_train, y_train = train_df[scaled_feature_cols], train_df["ClosePrice"]
X_test, y_test = test_df[scaled_feature_cols], test_df["ClosePrice"]

print(f"Training set: {X_train.shape[0]} rows | Test set: {X_test.shape[0]} rows")

Training set: 58295 rows | Test set: 11811 rows


In [11]:
# Train baseline Random Forest model
rf = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Train baseline XGBoost model
xgb_default = XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_default.fit(X_train, y_train)

rf_r2 = r2_score(y_test, rf.predict(X_test))
xgb_def_r2 = r2_score(y_test, xgb_default.predict(X_test))

print(f"Random Forest Test R2: {rf_r2:.4f}")
print(f"Default XGBoost Test R2: {xgb_def_r2:.4f}")

Random Forest Test R2: 0.6757
Default XGBoost Test R2: 0.6853


In [12]:
# Define hyperparameter grid for XGBoost
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.1, 1.0],
    'reg_lambda': [1.0, 5.0, 10.0]
}

xgb_base = XGBRegressor(random_state=42, n_jobs=-1)

# Randomized Search for hyperparameter tuning
xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=15,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

xgb_search.fit(X_train, y_train)

best_xgb = xgb_search.best_estimator_
tuned_r2 = r2_score(y_test, best_xgb.predict(X_test))

print("\n--- Hyperparameter Tuning Complete ---")
print("Best Parameters:", xgb_search.best_params_)
print(f"Tuned XGBoost Test R2 Score: {tuned_r2:.4f}")

Fitting 3 folds for each of 15 candidates, totalling 45 fits

--- Hyperparameter Tuning Complete ---
Best Parameters: {'subsample': 0.7, 'reg_lambda': 10.0, 'reg_alpha': 1.0, 'n_estimators': 300, 'max_depth': 10, 'learning_rate': 0.05, 'colsample_bytree': 0.7}
Tuned XGBoost Test R2 Score: 0.6905


In [13]:
# Summary Table
results_df = pd.DataFrame({
    "Model": ["Random Forest (Week 6)", "Default XGBoost", "Tuned XGBoost"],
    "Test R2 Score": [rf_r2, xgb_def_r2, tuned_r2]
})

print("=== Week 7 Model Comparison Summary ===")
print(results_df.to_string(index=False))

print("\n" + "="*40 + "\n")

# Extract feature importance from the tuned XGBoost model
importance_df = pd.DataFrame({
    "Feature": feature_cols_raw,
    "Importance": best_xgb.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("=== Top Drivers of California House Prices (XGBoost Feature Importance) ===")
print(importance_df.to_string(index=False))

=== Week 7 Model Comparison Summary ===
                 Model  Test R2 Score
Random Forest (Week 6)       0.675713
       Default XGBoost       0.685278
         Tuned XGBoost       0.690519


=== Top Drivers of California House Prices (XGBoost Feature Importance) ===
                  Feature  Importance
school_district_avg_price    0.323956
    BathroomsTotalInteger    0.286507
               LivingArea    0.184102
           bed_bath_ratio    0.069298
             property_age    0.059143
        LotSizeSquareFeet    0.042048
            BedroomsTotal    0.034945
